# 🎗️ Breast Cancer Detection V2
### Dataset: https://www.kaggle.com/datasets/aryashah2k/breast-ultrasound-images-dataset
### Classes: benign, malignant, normal
### Model: MobileNetV2 — 3 Class Classification with Class Weights

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile, json
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Upload to Drive: MyDrive/Datasets/Breast_cancer_detection.zip
zip_path = '/content/drive/MyDrive/Datasets/Breast_cancer_detection.zip'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/breast')
print('Extracted!')

# Find data dir — folder containing benign/malignant/normal
data_dir = None
for root, dirs, files in os.walk('/content/breast'):
    dl = [d.lower() for d in dirs]
    if any(x in dl for x in ['benign', 'malignant', 'normal']):
        data_dir = root
        break

print('Data dir:', data_dir)
print('Classes:', os.listdir(data_dir))
for cls in os.listdir(data_dir):
    print(f'  {cls}: {len(os.listdir(os.path.join(data_dir, cls)))} images')

In [ ]:
IMG_SIZE = 224
BATCH    = 16  # smaller batch — dataset chhota hai

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    validation_split=0.2
)

train_data = train_gen.flow_from_directory(
    data_dir, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH, class_mode='categorical',
    subset='training', seed=42
)
val_data = train_gen.flow_from_directory(
    data_dir, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH, class_mode='categorical',
    subset='validation', seed=42
)

# Expected: {'benign': 0, 'malignant': 1, 'normal': 2}
print('Classes:', train_data.class_indices)
print('Train:', train_data.samples, '| Val:', val_data.samples)

# Class weights — benign 437, malignant 210, normal 133
labels  = train_data.classes
weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weights = {i: w for i, w in enumerate(weights)}
print('Class weights:', class_weights)

In [ ]:
num_classes = len(train_data.class_indices)  # 3

base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x   = base(inp, training=False)
x   = layers.GlobalAveragePooling2D()(x)
x   = layers.Dense(256, activation='relu')(x)
x   = layers.Dropout(0.4)(x)
x   = layers.Dense(128, activation='relu')(x)
x   = layers.Dropout(0.3)(x)
out = layers.Dense(num_classes, activation='softmax')(x)

model = Model(inp, out)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# Phase 1: Train top layers
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.5, patience=2, monitor='val_loss', min_lr=1e-6)
]

history1 = model.fit(
    train_data, validation_data=val_data,
    epochs=10, callbacks=callbacks,
    class_weight=class_weights
)
print(f'Phase 1 Best: {max(history1.history["val_accuracy"])*100:.2f}%')

In [ ]:
# Phase 2: Fine-tune last 30 layers
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks2 = [
    EarlyStopping(patience=4, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.5, patience=2, monitor='val_loss', min_lr=1e-7)
]

history2 = model.fit(
    train_data, validation_data=val_data,
    epochs=10, callbacks=callbacks2,
    class_weight=class_weights
)
print(f'Phase 2 Best: {max(history2.history["val_accuracy"])*100:.2f}%')

In [ ]:
loss, acc = model.evaluate(val_data)
print(f'Val Accuracy: {acc*100:.2f}%')

preds        = model.predict(val_data)
pred_classes = np.argmax(preds, axis=1)
true_classes = val_data.classes
class_names  = list(val_data.class_indices.keys())

print('\nConfusion Matrix:')
print(confusion_matrix(true_classes, pred_classes))
print('\nClassification Report:')
print(classification_report(true_classes, pred_classes, target_names=class_names))

In [ ]:
save_dir = '/content/drive/MyDrive/ml_models'
os.makedirs(save_dir, exist_ok=True)

model.save(f'{save_dir}/breast_model.h5')
with open(f'{save_dir}/breast_classes.json', 'w') as f:
    json.dump(train_data.class_indices, f)

print('✅ breast_model.h5 saved!')
print('✅ breast_classes.json saved!')
print('Classes:', train_data.class_indices)